# TreeTaggger Evaluation on Latin Tokenisation, Lemmatisation, POS-tagging, and Processing Time
This notebook tests TreeTagger for tokenisation, lemmatization, and POS tagging on two datasets. Results are compared to the dataset's original tags.

## Results
Running the program generates two tables for each dataset which compares the predicted results with the gold standard. The table with the suffix all_detailed includes all tokens predicted by the model, while the table with the suffix aligned_only_detailed only contains results where the model tokenised the predicted results identically to the gold_standard. Additionally, a _summary.json file will be generated that compares the precision, accuracy, recall, and f1-scores between each dataset, as well as the time required to run the model for each dataset.  


## Structure:
- Import required libraries
- Configure variables
- Import and configure model
- function word_joiner(gold_df): Convert table of gold standard words into sentences for processing
- function treetagger_processor(sample_texts, sample_type): Run model on sentences
- function ttpos_to_upos(processed_results): Convert TreeTagger tags to UPOS (conversion generated with ChatGPT)
- function assessment(merged_df): Identify the similarity between gold_df and pred_df; adds new columns indicating whether lemma and POS results match, as well as a character-level similarity score between lemmas. 
- function analyse(df, sample_type, sample_version): Analyse results and generate f-scores, precision, and accuracy scores. 
- Run program
- Save results


Created by Thea Schaaf, April 2025

## Overview
The notebook takes a csv file of an annotated gold standard (gold_df) of each data type (glosses, medieval_charters) and reconstructs sentences from the words in each row with the function word_joiner(). The sentences are run through the nlp model via the function treetagger_processor(), which performs the actions of tokenisation, lemmatisation, and POS-tagging on each identified token. Because TreeTagger does not use UPOS-tags, the function ttpos_to_upos() is used to convert the produced tags to UPOS as accurately as possible. This data is returned as a dataframe (pred_df) which is merged with gold_df to create merged_df. The function assessment() adds new columns to the dataframe that indicate whether the gold_df and pred_df lemmas and POS-tags match as well as assign a character-similarity score between the lemmas. A dataframe without token misalignment, aligned_df, is generated. Finally, the function analyse() is run on merged_df and aligned_df to generate the accuracy, precision, recall, and f1-scores for each dataframe, and the results are saved. 

## Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
import difflib
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Configure variables

In [ ]:
# Configuration
MODEL_NAME = "TreeTagger"
SAMPLE_TYPES = ["medieval_charters", "glosses"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("04-treetagger.ipynb")

In [ ]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

## Import and configure model

In [2]:
import treetaggerwrapper
import re
from typing import List, Tuple, Dict, Any

/Users/Thea/nlplatin/lib/python3.10/site-packages/treetaggerwrapper.py:739: FutureWarning: Possible nested set at position 8
  punct2find_re = re.compile("([^ ])([[" + ALONEMARKS + "])",
/Users/Thea/nlplatin/lib/python3.10/site-packages/treetaggerwrapper.py:2043: FutureWarning: Possible nested set at position 152
  DnsHostMatch_re = re.compile("(" + DnsHost_expression + ")",
/Users/Thea/nlplatin/lib/python3.10/site-packages/treetaggerwrapper.py:2067: FutureWarning: Possible nested set at position 409
  UrlMatch_re = re.compile(UrlMatch_expression, re.VERBOSE | re.IGNORECASE)
/Users/Thea/nlplatin/lib/python3.10/site-packages/treetaggerwrapper.py:2079: FutureWarning: Possible nested set at position 192
  EmailMatch_re = re.compile(EmailMatch_expression, re.VERBOSE | re.IGNORECASE)


In [ ]:
 # Initialize TreeTagger with Latin model
treetagger_dir = "/Users/Thea/Downloads/tree-tagger-MacOSX-Intel-3.2.3" # Replace with own path
try:
    tagger = treetaggerwrapper.TreeTagger(
        TAGLANG='la',  # Latin language code
        TAGDIR=treetagger_dir,  # TreeTagger directory (optional)
        TAGOPT='-token -lemma -sgml -quiet'  # Options for tokenization and lemmatization
    )
    print("TreeTagger Latin processor initialized successfully.")
except Exception as e:
    raise RuntimeError(f"Failed to initialize TreeTagger: {e}")

TreeTagger Latin processor initialized successfully.


In [6]:
tagger = treetaggerwrapper.TreeTagger(TAGLANG='la')

## Load functions

In [7]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [8]:
def analyse(df, sample_type, df_version):
    # Convert the lemma columns to string type to ensure consistent comparison
    df['lemma_gold'] = df['lemma_gold'].astype(str)
    df['lemma_pred'] = df['lemma_pred'].astype(str)

    # Evaluate lemmatization
    lemma_accuracy = accuracy_score(df['lemma_gold'], df['lemma_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['lemma_gold'] == df['lemma_pred']

    # Fix the precision_recall_fscore_support call
    lemma_precision, lemma_recall, lemma_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    # Convert the lemma columns to string type to ensure consistent comparison
    df['pos_gold'] = df['pos_gold'].astype(str)
    df['pos_pred'] = df['pos_pred'].astype(str)

    # Evaluate lemmatization
    pos_accuracy = accuracy_score(df['pos_gold'], df['pos_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['pos_gold'] == df['pos_pred']

    # Fix the precision_recall_fscore_support call
    pos_precision, pos_recall, pos_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    results["accuracy"][f"{sample_type}_{df_version}_lemma"] = lemma_accuracy
    results["precision"][f"{sample_type}_{df_version}_lemma"] = lemma_precision
    results["recall"][f"{sample_type}_{df_version}_lemma"] = lemma_recall
    results["f1_score"][f"{sample_type}_{df_version}_lemma"] = lemma_f1

    results["accuracy"][f"{sample_type}_{df_version}_pos"] = pos_accuracy
    results["precision"][f"{sample_type}_{df_version}_pos"] = pos_precision
    results["recall"][f"{sample_type}_{df_version}_pos"] = pos_recall
    results["f1_score"][f"{sample_type}_{df_version}_pos"] = pos_f1

    df.to_csv(f"../results/{MODEL_NAME}_{sample_type}_{df_version}_detailed.csv", index=False)

    print(f"Completed {sample_type} {df_version}. Processing time: {processing_time:.2f}s")
    print(f"Lemmatization {df_version} accuracy: {lemma_accuracy:.4f}")
    print(f"POS tagging {df_version} accuracy: {pos_accuracy:.4f}")
    print("-" * 50)



In [9]:
def assessment(merged_df):

    # 1. POS match: exact string comparison
    merged_df['pos_match?'] = merged_df['pos_gold'] == merged_df['pos_pred']

    # 2. Lemma match: exact string comparison
    merged_df['lemma_match?'] = merged_df['lemma_gold'] == merged_df['lemma_pred']

    # 3. Lemma similarity: character-level similarity score (0 to 1)

    def lemma_similarity(a, b):
        return difflib.SequenceMatcher(None, str(a), str(b)).ratio()

    merged_df['lemma_sim?'] = merged_df.apply(
        lambda row: lemma_similarity(row['lemma_gold'], row['lemma_pred']), axis=1
    )

    return merged_df


In [10]:
def ttpos_to_upos(processed_results):
    pos_mappings = {
        'CC': 'CCONJ',          # Coordinating conjunction
        'CD': 'NUM',            # Cardinal number
        'DT': 'DET',            # Determiner
        'IN': 'ADP',            # Preposition
        'JJ': 'ADJ',            # Adjective
        'JJR': 'ADJ',           # Comparative adjective
        'JJS': 'ADJ',           # Superlative adjective
        'MD': 'AUX',            # Modal
        'NN': 'NOUN',           # Noun
        'NNS': 'NOUN',          # Plural noun
        'NNP': 'PROPN',         # Proper noun
        'NNPS': 'PROPN',        # Plural proper noun
        'PRP': 'PRON',          # Personal pronoun
        'PRP$': 'PRON',         # Possessive pronoun
        'POS': 'DET',           # Determiner
        'RB': 'ADV',            # Adverb
        'RBR': 'ADV',           # Comparative adverb
        'RBS': 'ADV',           # Superlative adverb
        'REL': 'DET',           # Relative pronoun
        'TO': 'PART',           # Particle 'to'
        'UH': 'INTJ',           # Interjection
        'VB': 'VERB',           # Verb base form
        'VBD': 'VERB',          # Past tense verb
        'VBG': 'VERB',          # Gerund/present participle
        'VBN': 'VERB',          # Past participle
        'VBP': 'VERB',          # Present tense verb
        'VBZ': 'VERB',          # 3rd person singular verb
        'WDT': 'DET',           # Wh-determiner
        'WP': 'PRON',           # Wh-pronoun
        'WRB': 'ADV',           # Wh-adverb
        'PUN': 'PUNCT',         # Punctuation
        'SENT': 'PUNCT',         # Punctuation
        'ADJ:NUM': 'NUM',        # Number
    }

    # Process each word dictionary
    updated_list = []
    for word_dict in processed_results:
        # Create a copy to avoid modifying original
        updated_dict = word_dict.copy()

        pos_tag = word_dict.get('pos', '')

        # Check exact matches first
        if pos_tag in pos_mappings:
            updated_dict['pos'] = pos_mappings[pos_tag]

        # Check pattern matches
        elif pos_tag.startswith('N:'):
            updated_dict['pos'] = 'NOUN'
        elif pos_tag.startswith('V:'):
            updated_dict['pos'] = 'VERB'
        elif pos_tag.startswith('A:'):
            updated_dict['pos'] = 'ADJ'
        elif pos_tag.startswith('ADJ:'):
            updated_dict['pos'] = 'ADJ'
        elif pos_tag.startswith('R:'):
            updated_dict['pos'] = 'ADV'
        elif pos_tag.startswith('P:'):
            updated_dict['pos'] = 'PRON'
        elif pos_tag.startswith('D:'):
            updated_dict['pos'] = 'DET'
        elif pos_tag.startswith('C:'):
            updated_dict['pos'] = 'CCONJ'
        elif pos_tag.startswith('I:'):
            updated_dict['pos'] = 'ADP'
        # Add more pattern rules as needed

        updated_list.append(updated_dict)

    return updated_list


In [16]:
def treetagger_processor(sample_texts, sample_type, treetagger_dir):

    tagger = treetaggerwrapper.TreeTagger(TAGLANG='la')

    processed_results = []

    for sample_id, text in sample_texts:
        # Tag the text
        tagged = tagger.tag_text(text)
        # Process each tagged word
        for word_id, line in enumerate(tagged):
            if '\t' in line:
                parts = line.split('\t')
                if len(parts) >= 3:
                    form = parts[0]
                    upos = parts[1]
                    lemma = parts[2]

                    if sample_type == "glosses":
                        word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{word_id}"
                    else:
                        word_id = str(word_id + 1)

                    processed_results.append({
                        "sample_id": sample_id,
                        "word_id": word_id,
                        "word": form,
                        "lemma": lemma,
                        "pos": upos
                    })

    return processed_results


## Run notebook

In [17]:
for sample_type in SAMPLE_TYPES:
    print(f"Processing {sample_type}...")

    # Load gold standard data
    gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

    gold_df = pd.read_csv(gold_file)

    # Extract text to reconstruct from words
    sample_texts = word_joiner(gold_df)

    # Process samples and measure time
    start_time = time.time()

    processed_results = treetagger_processor(sample_texts, sample_type, treetagger_dir)


    processing_time = time.time() - start_time
    results["processing_times"][sample_type] = processing_time

    print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

    upos_results = ttpos_to_upos(processed_results)

    #with open("demofile.txt", "a") as f:
        #f.write(str(upos_results))

    pred_df = pd.DataFrame(upos_results)

    # Add a token index per sample in both gold and pred dataframes
    gold_df['token_idx'] = gold_df.groupby('sample_id').cumcount()
    pred_df['token_idx'] = pred_df.groupby('sample_id').cumcount()

    # Merge on sample_id and token index
    merged_df = pd.merge(gold_df, pred_df, on=['sample_id', 'token_idx'], suffixes=('_gold', '_pred'))

    # Check mismatches
    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']

    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']
    mismatched_df = merged_df[~merged_df['word_match']].head()

    merged_df = assessment(merged_df)

    # Create df version without mismatches
    aligned_df = merged_df[merged_df['word_match']].copy()

    print(f"Running analysis on {sample_type}...")
    analyse(merged_df, sample_type, "all")
    analyse(aligned_df, sample_type, "aligned_only")






Processing medieval_charters...
Data processes with TreeTagger in 1.0531787872314453 seconds.
Running analysis on medieval_charters...
Completed medieval_charters all. Processing time: 1.05s
Lemmatization all accuracy: 0.7062
POS tagging all accuracy: 0.5909
--------------------------------------------------
Completed medieval_charters aligned_only. Processing time: 1.05s
Lemmatization aligned_only accuracy: 0.7231
POS tagging aligned_only accuracy: 0.6027
--------------------------------------------------
Processing glosses...
Data processes with TreeTagger in 0.7553939819335938 seconds.
Running analysis on glosses...
Completed glosses all. Processing time: 0.76s
Lemmatization all accuracy: 0.6862
POS tagging all accuracy: 0.5420
--------------------------------------------------
Completed glosses aligned_only. Processing time: 0.76s
Lemmatization aligned_only accuracy: 0.6914
POS tagging aligned_only accuracy: 0.5461
--------------------------------------------------
Processing class

## Save results

In [18]:
# Save summary results
with open(f"../results/{MODEL_NAME}_summary.json", "w") as f:
    json.dump(results, f, indent=2)